# Baseline models: training and evaluation

Trains and evaluates a TF-IDF baseline and two fine-tuned transformers (XLM-R, MuRIL) on the round-1 labelled data, tested on the 100-review gold set. Also runs SHAP explanations and a similar-review retrieval step.

## Setup

In [ ]:
!pip install -q shap sentence-transformers faiss-cpu
!nvidia-smi -L

## Load annotation files
Expects the gold set, the three per-annotator batches, and the two key files (star ratings, app names).

In [ ]:
from google.colab import files
uploaded = files.upload()

import glob
def find(prefix):
    hits = sorted(glob.glob(prefix + "*"))
    if not hits:
        raise FileNotFoundError(f"Could not find a file starting with '{prefix}'. Please upload it.")
    return hits[-1]
print("Found:", [find(p) for p in ["Gold_100", "Batch_Arpita", "Batch_Harsh", "Batch_Punit", "annotation_sample_key", "batch_key"]])

## Core utilities: loading, validation, baseline, scoring
Masks a batch of Overall labels that were annotated under an earlier guideline version, so they are excluded from training.

In [ ]:
import json
import random
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

SEED = 42
CATS = ["Delivery", "Price_Charges", "Offers_Discounts", "Product_Quality", "Availability",
        "Wrong_Missing_Item", "Refund_Return", "Payment_Wallet", "Customer_Support",
        "App_Experience", "Packaging", "Overall"]
LABELS = ["None", "Positive", "Negative", "Neutral"]
NAMES = ["Arpita", "Harsh", "Punit"]
# Labels we do NOT learn from: Punit's batch used the Overall category under the old (v1) rule,
# so those Overall labels are left out of training. Everything else from his batch is used.
MASK = {("Punit", "Overall")}


def usable(df, c):
    """Rows whose label for category c can be trained on."""
    return df[~df["source"].map(lambda s: (s, c) in MASK)]


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)


def _norm(v):
    if pd.isna(v) or str(v).strip() == "":
        return "None"
    v = str(v).strip().capitalize()
    return v


def read_sheet(path, source):
    d = pd.read_excel(path, sheet_name="Annotate")
    d = d[d["ID"].astype(str) != "EXAMPLE"].copy()
    missing = [c for c in CATS if c not in d.columns]
    if missing:
        raise ValueError(f"{path}: missing columns {missing}. Please don't rename or delete columns.")
    problems = []
    for c in CATS:
        d[c] = d[c].apply(_norm)
        bad = d[~d[c].isin(LABELS)]
        for _, row in bad.iterrows():
            fix = "Neutral" if row[c] == "Mixed" else "None"
            problems.append(f"{path} | {row['ID']} | {c} = '{row[c]}' -> changed to {fix}")
        # rule v2: mixed feeling inside one category counts as Neutral; anything else unknown is dropped
        d[c] = d[c].where(d[c].isin(LABELS), d[c].map(lambda v: "Neutral" if v == "Mixed" else "None"))
    d["Review_Sentiment"] = d["Review_Sentiment"].apply(_norm) if "Review_Sentiment" in d.columns else "None"
    notes = d["Notes"] if "Notes" in d.columns else pd.Series("", index=d.index)
    d["skip"] = notes.astype(str).str.strip().str.lower().eq("skip")
    d["n_labels"] = (d[CATS] != "None").sum(axis=1)
    d["source"] = source
    return d, problems


def load_all(gold_path, batch_paths, gold_key, batch_key):
    gold, p1 = read_sheet(gold_path, "gold")
    batches, probs = [], list(p1)
    for n, p in batch_paths.items():
        b, pb = read_sheet(p, n)
        batches.append(b)
        probs += pb
    batch = pd.concat(batches, ignore_index=True)

    report = {"invalid_values": probs}
    for name, d in (("gold", gold), ("batch", batch)):
        empty = d[(d.n_labels == 0) & (~d.skip)]
        report[f"{name}_unlabelled_rows"] = empty["ID"].tolist()
        report[f"{name}_skipped"] = int(d.skip.sum())

    keys = pd.concat([pd.read_csv(gold_key), pd.read_csv(batch_key)], ignore_index=True)
    keys = keys.rename(columns={"id": "ID"})[["ID", "stars", "review_id"]]
    cols = ["ID", "App", "Review", "Review_Sentiment", "source"] + CATS
    data = pd.concat([gold[~gold.skip], batch[~batch.skip]], ignore_index=True)[cols]
    data = data.merge(keys, on="ID", how="left")
    # rows with no aspect label at all are dropped (they were probably not labelled)
    data = data[(data[CATS] != "None").any(axis=1)].reset_index(drop=True)
    return data, report


def make_splits(data, val_size=0.15):
    test = data[data.source == "gold"].reset_index(drop=True)
    rest = data[data.source != "gold"]
    train, val = train_test_split(rest, test_size=val_size, random_state=SEED)
    return train.reset_index(drop=True), val.reset_index(drop=True), test


def label_stats(train, val, test):
    rows = []
    for c in CATS:
        row = {"Category": c}
        for name, d in (("Train", train), ("Val", val), ("Test", test)):
            vc = d[c].value_counts()
            row[f"{name} mentions"] = int(len(d) - vc.get("None", 0))
        for s in ["Positive", "Negative", "Neutral"]:
            row[f"All {s}"] = int(sum((d[c] == s).sum() for d in (train, val, test)))
        rows.append(row)
    return pd.DataFrame(rows)


# ------------------------------------------------------------------ metrics
def _prf(pred, gold):
    tp = len(pred & gold)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def evaluate(gold_df, pred_df, cats=None):
    """Both dataframes: one row per review (same order), one column per category with a label."""
    g_det, p_det, g_full, p_full = set(), set(), set(), set()
    per_cat = []
    for c in (cats or CATS):
        g_c, p_c = set(), set()
        for i in range(len(gold_df)):
            g, p = gold_df.iloc[i][c], pred_df.iloc[i][c]
            rid = gold_df.iloc[i]["ID"]
            if g != "None":
                g_det.add((rid, c)); g_full.add((rid, c, g)); g_c.add((rid, g))
            if p != "None":
                p_det.add((rid, c)); p_full.add((rid, c, p)); p_c.add((rid, p))
        if g_c:
            per_cat.append((c, _prf(p_c, g_c)[2], len(g_c)))
    dp, dr, df = _prf(p_det, g_det)
    fp, fr, ff = _prf(p_full, g_full)
    macro = float(np.mean([f for _, f, _ in per_cat])) if per_cat else 0.0
    return {
        "Detection P": round(dp, 3), "Detection R": round(dr, 3), "Detection F1": round(df, 3),
        "Aspect+Sentiment P": round(fp, 3), "Aspect+Sentiment R": round(fr, 3),
        "Aspect+Sentiment F1": round(ff, 3), "Macro F1 (per category)": round(macro, 3),
    }, pd.DataFrame(per_cat, columns=["Category", "F1", "Gold mentions"])


# ------------------------------------------------------------------ baseline
def tfidf_baseline(train, test):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=1, lowercase=True)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, lowercase=True)
    Xtr = hstack([word.fit_transform(train.Review), char.fit_transform(train.Review)]).tocsr()
    Xte = hstack([word.transform(test.Review), char.transform(test.Review)]).tocsr()
    pred = test[["ID"]].copy()
    for c in CATS:
        keep = train.index.isin(usable(train, c).index)
        y = train[c].values[keep]
        if len(set(y)) < 2:
            pred[c] = y[0]
            continue
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
        clf.fit(Xtr[keep], y)
        pred[c] = clf.predict(Xte)
    return pred


def export(path, data, splits, metrics, per_cat_tables, predictions, explanations, neighbours, stats):
    train, val, test = splits
    split_of = {**{i: "train" for i in train.ID}, **{i: "val" for i in val.ID}, **{i: "test" for i in test.ID}}
    reviews = []
    for _, r in data.iterrows():
        rid = r["ID"]
        item = {"id": rid, "app": r["App"], "stars": None if pd.isna(r["stars"]) else int(r["stars"]),
                "text": r["Review"], "split": split_of.get(rid, "?"),
                "human": {c: r[c] for c in CATS if r[c] != "None"},
                "review_sentiment": r["Review_Sentiment"]}
        if rid in predictions:
            item["model"] = predictions[rid]
        if rid in explanations:
            item["explanation"] = explanations[rid]
        if rid in neighbours:
            item["similar"] = neighbours[rid]
        reviews.append(item)
    out = {"categories": CATS, "metrics": metrics,
           "per_category": {k: v.to_dict(orient="records") for k, v in per_cat_tables.items()},
           "label_stats": stats.to_dict(orient="records"), "reviews": reviews}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=1)


## Load and validate labels
Checks for invalid label values and rows with no annotation.

In [ ]:
set_seed()
data, report = load_all(find("Gold_100"),
                        {n: find(f"Batch_{n}") for n in NAMES},
                        find("annotation_sample_key"), find("batch_key"))
print("Reviews ready for training/testing:", len(data))
for k, v in report.items():
    print(f"{k}: {v if not isinstance(v, list) else (len(v), v[:10])}")

## Train/validation/test split and label statistics

In [ ]:
train, val, test = make_splits(data)
print(f"Train: {len(train)}   Validation: {len(val)}   Test (gold): {len(test)}\n")
stats = label_stats(train, val, test)
print(stats.to_string(index=False))

## Baseline: TF-IDF + logistic regression
Word bigrams and character n-grams (to partly capture spelling variants), one class-balanced classifier per category.

In [ ]:
results, per_cat = {}, {}
pred_tfidf = tfidf_baseline(pd.concat([train, val]), test)
results["TF-IDF + LR"], per_cat["TF-IDF + LR"] = evaluate(test, pred_tfidf)
print(results["TF-IDF + LR"])

## Transformer utilities
Sentence-pair formulation following Sun et al. (2019): (review, category name) -> sentiment.

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
L2I = {l: i for i, l in enumerate(LABELS)}
print("Using:", DEVICE)


def cat_text(c):
    return c.replace("_", " ").lower()          # "Price_Charges" -> "price charges"


def to_pairs(df, none_ratio=3, seed=SEED):
    """One example per (review, category). Keeps all real aspects and at most
    `none_ratio` times as many 'None' examples, so the model is not flooded by 'None'."""
    rows = [(r["Review"], cat_text(c), L2I[r[c]]) for _, r in df.iterrows() for c in CATS
            if (r["source"], c) not in MASK]
    if none_ratio:
        pos = [x for x in rows if x[2] != 0]
        neg = [x for x in rows if x[2] == 0]
        rng = random.Random(seed)
        rows = pos + rng.sample(neg, min(len(neg), none_ratio * len(pos)))
        rng.shuffle(rows)
    return rows


@torch.no_grad()
def predict_proba(model, tok, texts, cat, max_len=128, bs=64):
    model.eval()
    out = []
    texts = [str(t) for t in texts]
    for i in range(0, len(texts), bs):
        chunk = texts[i:i + bs]
        enc = tok(chunk, [cat_text(cat)] * len(chunk), truncation=True, max_length=max_len,
                  padding=True, return_tensors="pt").to(DEVICE)
        out.append(torch.softmax(model(**enc).logits, -1).float().cpu().numpy())
    return np.concatenate(out)


def predict_transformer(model, tok, df):
    pred = df[["ID"]].copy()
    for c in CATS:
        p = predict_proba(model, tok, df.Review.tolist(), c)
        pred[c] = [LABELS[i] for i in p.argmax(1)]
    return pred


def train_transformer(model_name, train, val, epochs=12, lr=3e-5, bs=16, max_len=128):
    set_seed()
    torch.manual_seed(SEED)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(LABELS)).to(DEVICE)
    rows = to_pairs(train)

    def collate(batch):
        enc = tok([b[0] for b in batch], [b[1] for b in batch], truncation=True,
                  max_length=max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([b[2] for b in batch])
        return enc

    dl = DataLoader(rows, batch_size=bs, shuffle=True, collate_fn=collate,
                    generator=torch.Generator().manual_seed(SEED))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * len(dl)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    best_f1, best_state, log = -1, None, []
    for ep in range(epochs):
        model.train()
        total = 0.0
        for b in dl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            loss = model(**b).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
            total += loss.item()
        m, _ = evaluate(val, predict_transformer(model, tok, val))
        log.append({"model": model_name, "epoch": ep + 1, "train_loss": round(total / len(dl), 4),
                    "val_F1": m["Aspect+Sentiment F1"]})
        print(log[-1])
        if m["Aspect+Sentiment F1"] > best_f1:
            best_f1 = m["Aspect+Sentiment F1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, tok, log


## Fine-tune XLM-R

In [ ]:
xlmr, xlmr_tok, log_xlmr = train_transformer("xlm-roberta-base", train, val)
pred_xlmr = predict_transformer(xlmr, xlmr_tok, test)
results["XLM-R"], per_cat["XLM-R"] = evaluate(test, pred_xlmr)
print(results["XLM-R"])

## Fine-tune MuRIL
Higher learning rate than XLM-R (5e-5) -- an earlier run at 2e-5 did not converge.

In [ ]:
muril, muril_tok, log_muril = train_transformer("google/muril-base-cased", train, val, lr=5e-5)
pred_muril = predict_transformer(muril, muril_tok, test)
results["MuRIL"], per_cat["MuRIL"] = evaluate(test, pred_muril)
print(results["MuRIL"])

## Model comparison
Best transformer selected by validation score, not test score.

In [ ]:
table = pd.DataFrame(results).T
print("=== All 12 categories ===")
print(table.to_string())
NO_OVERALL = [c for c in CATS if c != "Overall"]
preds_by_model = {"TF-IDF + LR": pred_tfidf, "XLM-R": pred_xlmr, "MuRIL": pred_muril}
table_no_overall = pd.DataFrame({k: evaluate(test, v, NO_OVERALL)[0] for k, v in preds_by_model.items()}).T
print("\n=== Without the Overall category (annotators applied it inconsistently) ===")
print(table_no_overall.to_string())
best_val = {"XLM-R": max(l["val_F1"] for l in log_xlmr), "MuRIL": max(l["val_F1"] for l in log_muril)}
BEST = max(best_val, key=best_val.get)
model, tok, pred_best = (xlmr, xlmr_tok, pred_xlmr) if BEST == "XLM-R" else (muril, muril_tok, pred_muril)
print("\nBest transformer (by validation):", BEST, best_val)
print("\nPer-category F1 for", BEST)
print(per_cat[BEST].to_string(index=False))

## SHAP setup

In [ ]:
import shap


def _merge_words(pieces, values):
    """SHAP returns sub-word pieces; join them back into whole words and add up their scores."""
    words, new_word = [], True
    for p, v in zip(pieces, values):
        if not p.strip():
            new_word = True
            continue
        if new_word or p[:1].isspace() or not words:
            words.append([p.strip(), float(v)])
        else:
            words[-1][0] += p.strip()
            words[-1][1] += float(v)
        new_word = p[-1:].isspace()      # a piece ending in a space closes the word
    return [(w, round(v, 4)) for w, v in words]


def occlusion(model, tok, text, cat, idx):
    """Backup method: remove one word at a time and see how much the probability drops."""
    words = text.split()
    base = predict_proba(model, tok, [text], cat)[0, idx]
    variants = [" ".join(words[:i] + words[i + 1:]) for i in range(len(words))]
    probs = predict_proba(model, tok, variants, cat)[:, idx]
    return [(w, round(float(base - p), 4)) for w, p in zip(words, probs)]


def explain_reviews(model, tok, df, preds):
    """For each review and each aspect the model found, score every word."""
    out, method_used = {}, set()
    masker = shap.maskers.Text(tok)
    for i, r in df.iterrows():
        rid, text = r["ID"], r["Review"]
        found = [(c, preds.iloc[i][c]) for c in CATS if preds.iloc[i][c] != "None"]
        items = []
        for c, label in found:
            idx = L2I[label]
            try:
                f = lambda xs, c=c: predict_proba(model, tok, list(xs), c)
                sv = shap.Explainer(f, masker, output_names=LABELS)([text])
                words = _merge_words(list(sv.data[0]), sv.values[0][:, idx])
                method_used.add("SHAP")
            except Exception as e:
                words = occlusion(model, tok, text, c, idx)
                method_used.add("occlusion")
            items.append({"category": c, "label": label, "words": words})
        out[rid] = items
    print("Explanation method used:", method_used or "none")
    return out


def similar_reviews(data, past_ids, k=3):
    """Retrieval step of RAG: for every review, find the most similar labelled past reviews."""
    from sentence_transformers import SentenceTransformer
    import faiss
    enc = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    E = enc.encode(data.Review.tolist(), normalize_embeddings=True, batch_size=64,
                   show_progress_bar=True).astype("float32")
    past_mask = data.ID.isin(past_ids).values
    past_idx = np.where(past_mask)[0]
    index = faiss.IndexFlatIP(E.shape[1])
    index.add(E[past_idx])
    sims, nn = index.search(E, k + 1)
    out = {}
    for i in range(len(data)):
        items = []
        for s, j in zip(sims[i], nn[i]):
            row = data.iloc[past_idx[j]]
            if row["ID"] == data.iloc[i]["ID"]:
                continue
            items.append({"id": row["ID"], "text": row["Review"], "similarity": round(float(s), 3),
                          "human": {c: row[c] for c in CATS if row[c] != "None"}})
        out[data.iloc[i]["ID"]] = items[:k]
    return out


## Explain test predictions
Per-word contribution scores for each (review, predicted aspect) pair.

In [ ]:
explanations = explain_reviews(model, tok, test, pred_best)
first = next(iter(explanations.items()))
print(first[0], test.set_index("ID").loc[first[0], "Review"])
for item in first[1]:
    top = sorted(item["words"], key=lambda w: -w[1])[:5]
    print(" ", item["category"], item["label"], "| top words:", top)

## Retrieval
Embeds reviews with a multilingual sentence encoder and indexes them with FAISS; returns the k nearest labelled reviews for each query.

In [ ]:
past_ids = set(train.ID) | set(val.ID)
neighbours = similar_reviews(data, past_ids, k=3)
rid = test.ID.iloc[0]
print("Review:", test.Review.iloc[0])
for n in neighbours[rid]:
    print("  similar:", n["similarity"], "|", n["text"][:90], "|", n["human"])

## Export results and metrics

In [ ]:
all_pred = predict_transformer(model, tok, data)
predictions = {r.ID: {c: r[c] for c in CATS if r[c] != "None"} for _, r in all_pred.iterrows()}
metrics = {k: v for k, v in results.items()}
metrics["best_model"] = BEST
table.to_csv("metrics.csv")
table_no_overall.to_csv("metrics_without_overall.csv")
metrics["without_overall"] = table_no_overall.to_dict(orient="index")
export("results.json", data, (train, val, test), metrics, per_cat, predictions, explanations, neighbours, stats)
pd.DataFrame(log_xlmr + log_muril).to_csv("training_log.csv", index=False)
for f in ["results.json", "metrics.csv", "metrics_without_overall.csv", "training_log.csv"]:
    files.download(f)